# 01 · Datos y esquema

## 1 · Verificar el entorno

In [3]:
import sys
from pathlib import Path
import pyspark

print("Python:", sys.version.split()[0])
print("PySpark:", pyspark.__version__)
print("Carpeta del kernel:", Path.cwd())

Python: 3.13.15
PySpark: 4.2.0
Carpeta del kernel: /home/jovyan/work/spark/notebooks


## 2 · Localizar los datos

In [17]:
import os
from pathlib import Path

datos = Path(os.environ["AIRLINE_LOCAL_DATA_DIR"])
print("Carpeta de datos:", datos)
print("Existe:", datos.is_dir())
print("Archivos:", sorted(archivo.name for archivo in datos.iterdir()))
archivo_1997 = datos / "1997.csv" / "1997.csv"
print("CSV de 1997 encontrado:", archivo_1997.is_file())

Carpeta de datos: /home/jovyan/datasets
Existe: True
Archivos: ['1987.csv', '1988.csv', '1989.csv', '1990.csv', '1991.csv', '1992.csv', '1993.csv', '1994.csv', '1995.csv', '1996.csv', '1997.csv', '1998.csv', '1999.csv', '2000.csv', '2001.csv', '2002.csv', '2003.csv', '2004.csv', '2005.csv', '2006.csv', '2007.csv', '2008.csv', 'airports.csv', 'carriers.csv', 'plane-data.csv', 'variable-descriptions.csv']
CSV de 1997 encontrado: True


## 3 · Inspeccionar el CSV de vuelos

In [5]:
import csv

with (datos / "1997.csv").open(newline="", encoding="utf-8") as archivo:
    lector = csv.reader(archivo)
    columnas = next(lector)
    primera_fila = next(lector)

print("Número de columnas:", len(columnas))
print("Nombres:", columnas)
print("Primer vuelo:", dict(zip(columnas, primera_fila)))

Número de columnas: 29
Nombres: ['Year', 'Month', 'DayofMonth', 'DayOfWeek', 'DepTime', 'CRSDepTime', 'ArrTime', 'CRSArrTime', 'UniqueCarrier', 'FlightNum', 'TailNum', 'ActualElapsedTime', 'CRSElapsedTime', 'AirTime', 'ArrDelay', 'DepDelay', 'Origin', 'Dest', 'Distance', 'TaxiIn', 'TaxiOut', 'Cancelled', 'CancellationCode', 'Diverted', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']
Primer vuelo: {'Year': '1997', 'Month': '1', 'DayofMonth': '28', 'DayOfWeek': '2', 'DepTime': '1615', 'CRSDepTime': '1615', 'ArrTime': '1728', 'CRSArrTime': '1725', 'UniqueCarrier': 'WN', 'FlightNum': '1159', 'TailNum': 'N683', 'ActualElapsedTime': '73', 'CRSElapsedTime': '70', 'AirTime': '60', 'ArrDelay': '3', 'DepDelay': '0', 'Origin': 'ABQ', 'Dest': 'PHX', 'Distance': '328', 'TaxiIn': '4', 'TaxiOut': '9', 'Cancelled': '0', 'CancellationCode': 'NA', 'Diverted': '0', 'CarrierDelay': 'NA', 'WeatherDelay': 'NA', 'NASDelay': 'NA', 'SecurityDelay': 'NA', 'LateAircraftDelay': '

## 4 · Inspeccionar aeropuertos y aerolíneas

In [6]:
for nombre in ("airports.csv", "carriers.csv"):
    with (datos / nombre).open(newline="", encoding="utf-8") as archivo:
        lector = csv.reader(archivo)
        columnas = next(lector)
        primera_fila = next(lector)

    print(nombre)
    print("Columnas:", columnas)
    print("Primera fila:", dict(zip(columnas, primera_fila)))

airports.csv
Columnas: ['iata', 'airport', 'city', 'state', 'country', 'lat', 'long']
Primera fila: {'iata': '00M', 'airport': 'Thigpen ', 'city': 'Bay Springs', 'state': 'MS', 'country': 'USA', 'lat': '31.95376472', 'long': '-89.23450472'}
carriers.csv
Columnas: ['Code', 'Description']
Primera fila: {'Code': '02Q', 'Description': 'Titan Airways'}


## 5 · Definir el esquema de vuelos

In [7]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

campos_enteros = {
    "Year", "Month", "DayofMonth", "DayOfWeek",
    "ActualElapsedTime", "CRSElapsedTime", "AirTime",
    "ArrDelay", "DepDelay", "Distance", "TaxiIn", "TaxiOut",
    "Cancelled", "Diverted", "CarrierDelay", "WeatherDelay",
    "NASDelay", "SecurityDelay", "LateAircraftDelay",
}

campos_texto = {
    "DepTime", "CRSDepTime", "ArrTime", "CRSArrTime",
    "UniqueCarrier", "FlightNum", "TailNum",
    "Origin", "Dest", "CancellationCode",
}

with (datos / "1997.csv").open(newline="", encoding="utf-8") as archivo:
    nombres_vuelos = next(csv.reader(archivo))

assert set(nombres_vuelos) == campos_enteros | campos_texto

schema_vuelos = StructType([
    StructField(
        nombre,
        IntegerType() if nombre in campos_enteros else StringType(),
        nullable=True,
    )
    for nombre in nombres_vuelos
])

for campo in schema_vuelos:
    print(campo.name, campo.dataType.simpleString())

Year int
Month int
DayofMonth int
DayOfWeek int
DepTime string
CRSDepTime string
ArrTime string
CRSArrTime string
UniqueCarrier string
FlightNum string
TailNum string
ActualElapsedTime int
CRSElapsedTime int
AirTime int
ArrDelay int
DepDelay int
Origin string
Dest string
Distance int
TaxiIn int
TaxiOut int
Cancelled int
CancellationCode string
Diverted int
CarrierDelay int
WeatherDelay int
NASDelay int
SecurityDelay int
LateAircraftDelay int


### Comprobar la codificación de todos los años

In [18]:
import codecs

anios_con_error = []

for anio, archivo in sorted(archivos_por_anio.items()):
    decodificador = codecs.getincrementaldecoder("utf-8")(errors="strict")

    try:
        with archivo.open("rb") as contenido:
            while bloque := contenido.read(1024 * 1024):
                decodificador.decode(bloque)

            decodificador.decode(b"", final=True)

    except UnicodeDecodeError:
        anios_con_error.append(anio)
        print(f"{anio}: contiene bytes que no son UTF-8")
    else:
        print(f"{anio}: UTF-8 OK")

print("Años con problemas de codificación:", anios_con_error)

1987: UTF-8 OK
1988: UTF-8 OK
1989: UTF-8 OK
1990: UTF-8 OK
1991: UTF-8 OK
1992: UTF-8 OK
1993: UTF-8 OK
1994: UTF-8 OK
1995: UTF-8 OK
1996: UTF-8 OK
1997: UTF-8 OK
1998: UTF-8 OK
1999: UTF-8 OK
2000: UTF-8 OK
2001: contiene bytes que no son UTF-8
2002: contiene bytes que no son UTF-8
2003: UTF-8 OK
2004: UTF-8 OK
2005: UTF-8 OK
2006: UTF-8 OK
2007: UTF-8 OK
2008: UTF-8 OK
Años con problemas de codificación: [2001, 2002]


### Contar filas con problemas de codificación

In [19]:
resumen_codificacion = {}

for anio in anios_con_error:
    total_filas = 0
    filas_con_error = 0
    primeras_lineas = []

    with archivos_por_anio[anio].open("rb") as contenido:
        next(contenido)  # Saltar la cabecera

        for numero_linea, linea in enumerate(contenido, start=2):
            total_filas += 1

            try:
                linea.decode("utf-8")
            except UnicodeDecodeError:
                filas_con_error += 1

                if len(primeras_lineas) < 3:
                    primeras_lineas.append(numero_linea)

    resumen_codificacion[anio] = {
        "total": total_filas,
        "con_error": filas_con_error,
    }

    porcentaje = 100 * filas_con_error / total_filas
    print(
        f"{anio}: {filas_con_error:,} de {total_filas:,} filas "
        f"({porcentaje:.6f}%)"
    )
    print("Primeras líneas afectadas:", primeras_lineas)

2001: 2,545,013 de 5,967,780 filas (42.645892%)
Primeras líneas afectadas: [2, 3, 4]
2002: 335,925 de 5,271,359 filas (6.372645%)
Primeras líneas afectadas: [2, 3, 4]


### Examinar ejemplos de 2001 y 2002

In [20]:
for anio in anios_con_error:
    encontrados = 0
    print(f"\nAño {anio}")

    with archivos_por_anio[anio].open("rb") as contenido:
        next(contenido)  # Cabecera

        for numero_linea, linea in enumerate(contenido, start=2):
            try:
                linea.decode("utf-8")
            except UnicodeDecodeError:
                texto = linea.decode(
                    "utf-8", errors="surrogateescape"
                ).rstrip("\r\n")
                valores = next(csv.reader([texto]))

                print(
                    f"Línea {numero_linea}: "
                    f"{len(valores)} columnas"
                )

                for nombre, valor in zip(nombres_vuelos, valores):
                    tiene_error = any(
                        0xDC80 <= ord(caracter) <= 0xDCFF
                        for caracter in valor
                    )
                    if tiene_error:
                        print(f"  {nombre}: {valor!r}")

                encontrados += 1
                if encontrados == 3:
                    break


Año 2001
Línea 2: 29 columnas
  TailNum: 'N700\udce4\udce6'
Línea 3: 29 columnas
  TailNum: 'N713\udce4\udce6'
Línea 4: 29 columnas
  TailNum: 'N702\udce4\udce6'

Año 2002
Línea 2: 29 columnas
  TailNum: 'N709\udce4\udce6'
Línea 3: 29 columnas
  TailNum: 'N733\udce4\udce6'
Línea 4: 29 columnas
  TailNum: 'N758\udce4\udce6'


### Contar las columnas afectadas por codificación

In [21]:
from collections import Counter

for anio in anios_con_error:
    por_columna = Counter()
    filas_con_error = 0
    solo_tailnum = 0
    otras_columnas = 0
    estructura_incorrecta = 0
    ejemplos_otros = []

    with archivos_por_anio[anio].open("rb") as contenido:
        next(contenido)  # Cabecera

        for numero_linea, linea in enumerate(contenido, start=2):
            try:
                linea.decode("utf-8")
                continue
            except UnicodeDecodeError:
                filas_con_error += 1

            texto = linea.decode(
                "utf-8", errors="surrogateescape"
            ).rstrip("\r\n")
            valores = next(csv.reader([texto]))

            if len(valores) != len(nombres_vuelos):
                estructura_incorrecta += 1
                continue

            afectadas = [
                nombre
                for nombre, valor in zip(nombres_vuelos, valores)
                if any(
                    0xDC80 <= ord(caracter) <= 0xDCFF
                    for caracter in valor
                )
            ]

            por_columna.update(afectadas)

            if afectadas == ["TailNum"]:
                solo_tailnum += 1
            else:
                otras_columnas += 1
                if len(ejemplos_otros) < 3:
                    ejemplos_otros.append((numero_linea, afectadas))

    print(f"\nAño {anio}")
    print("Filas con error UTF-8:", f"{filas_con_error:,}")
    print("Solo TailNum:", f"{solo_tailnum:,}")
    print("Otras columnas:", f"{otras_columnas:,}")
    print("Número de columnas distinto de 29:", estructura_incorrecta)
    print("Recuento por columna:", por_columna.most_common())
    print("Primeros ejemplos fuera de TailNum:", ejemplos_otros)


Año 2001
Filas con error UTF-8: 2,545,013
Solo TailNum: 2,545,013
Otras columnas: 0
Número de columnas distinto de 29: 0
Recuento por columna: [('TailNum', 2545013)]
Primeros ejemplos fuera de TailNum: []

Año 2002
Filas con error UTF-8: 335,925
Solo TailNum: 335,925
Otras columnas: 0
Número de columnas distinto de 29: 0
Recuento por columna: [('TailNum', 335925)]
Primeros ejemplos fuera de TailNum: []


## 6 · Leer vuelos con Spark

In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Airline ETL local").getOrCreate()

flights = spark.read.csv(
    str(datos / "1997.csv"),
    schema=schema_vuelos,
    header=True,
    nullValue="NA",
    enforceSchema=False,
)

print("Ejecución:", spark.sparkContext.master)
flights.select(
    "Year", "UniqueCarrier", "Origin", "Dest",
    "ArrDelay", "Cancelled", "Diverted"
).show(5, truncate=False)

Ejecución: local[2]
+----+-------------+------+----+--------+---------+--------+
|Year|UniqueCarrier|Origin|Dest|ArrDelay|Cancelled|Diverted|
+----+-------------+------+----+--------+---------+--------+
|1997|WN           |ABQ   |PHX |3       |0        |0       |
|1997|WN           |ABQ   |PHX |10      |0        |0       |
|1997|WN           |ABQ   |PHX |5       |0        |0       |
|1997|WN           |ABQ   |PHX |4       |0        |0       |
|1997|WN           |ABQ   |PHX |44      |0        |0       |
+----+-------------+------+----+--------+---------+--------+
only showing top 5 rows


Comprobar para todos los anos

In [16]:
import csv

archivos_por_anio = {}

for anio in range(1987, 2009):
    carpeta = datos / f"{anio}.csv"

    if not carpeta.is_dir():
        print(f"{anio}: falta la carpeta")
        continue

    csvs = sorted(archivo for archivo in carpeta.rglob("*.csv") if archivo.is_file())

    if len(csvs) != 1:
        print(f"{anio}: encontré {len(csvs)} archivos CSV; esperaba 1")
        continue

    archivos_por_anio[anio] = csvs[0]

def leer_cabecera(archivo):
    with archivo.open("rb") as contenido:
        primera_linea = contenido.readline()

    texto = primera_linea.decode("utf-8-sig").rstrip("\r\n")
    return next(csv.reader([texto]))


nombres_vuelos = leer_cabecera(archivos_por_anio[1997])

for anio, archivo in sorted(archivos_por_anio.items()):
    cabecera = leer_cabecera(archivo)
    resultado = "OK" if cabecera == nombres_vuelos else "DIFERENTE"
    print(f"{anio}: {resultado} — {archivo.relative_to(datos)}")

print("Años comprobados:", len(archivos_por_anio), "de 22")

1987: OK — 1987.csv/1987.csv
1988: OK — 1988.csv/1988.csv
1989: OK — 1989.csv/1989.csv
1990: OK — 1990.csv/1990.csv
1991: OK — 1991.csv/1991.csv
1992: OK — 1992.csv/1992.csv
1993: OK — 1993.csv/1993.csv
1994: OK — 1994.csv/1994.csv
1995: OK — 1995.csv/1995.csv
1996: OK — 1996.csv/1996.csv
1997: OK — 1997.csv/1997.csv
1998: OK — 1998.csv/1998.csv
1999: OK — 1999.csv/1999.csv
2000: OK — 2000.csv/2000.csv
2001: OK — 2001.csv/2001.csv
2002: OK — 2002.csv/2002.csv
2003: OK — 2003.csv/2003.csv
2004: OK — 2004.csv/2004.csv
2005: OK — 2005.csv/2005.csv
2006: OK — 2006.csv/2006.csv
2007: OK — 2007.csv/2007.csv
2008: OK — 2008.csv/2008.csv
Años comprobados: 22 de 22


Definir esquemas para airports y carriers

In [9]:
from pyspark.sql.types import DoubleType

schema_airports = StructType([
    StructField("iata", StringType(), True),
    StructField("airport", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("long", DoubleType(), True),
])

schema_carriers = StructType([
    StructField("Code", StringType(), True),
    StructField("Description", StringType(), True),
])

for nombre, esquema in (
    ("airports.csv", schema_airports),
    ("carriers.csv", schema_carriers),
):
    with (datos / nombre).open(newline="", encoding="utf-8") as archivo:
        cabecera = next(csv.reader(archivo))

    assert cabecera == esquema.fieldNames()
    print(nombre, esquema.simpleString())

airports.csv struct<iata:string,airport:string,city:string,state:string,country:string,lat:double,long:double>
carriers.csv struct<Code:string,Description:string>


Leer tablas aux

In [10]:
airports = spark.read.csv(
    str(datos / "airports.csv"),
    schema=schema_airports,
    header=True,
    nullValue="NA",
    enforceSchema=False,
)

carriers = spark.read.csv(
    str(datos / "carriers.csv"),
    schema=schema_carriers,
    header=True,
    nullValue="NA",
    enforceSchema=False,
)

airports.show(5, truncate=False)
carriers.show(5, truncate=False)

+----+--------------------+----------------+-----+-------+-----------+------------+
|iata|airport             |city            |state|country|lat        |long        |
+----+--------------------+----------------+-----+-------+-----------+------------+
|00M |Thigpen             |Bay Springs     |MS   |USA    |31.95376472|-89.23450472|
|00R |Livingston Municipal|Livingston      |TX   |USA    |30.68586111|-95.01792778|
|00V |Meadow Lake         |Colorado Springs|CO   |USA    |38.94574889|-104.5698933|
|01G |Perry-Warsaw        |Perry           |NY   |USA    |42.74134667|-78.05208056|
|01J |Hilliard Airpark    |Hilliard        |FL   |USA    |30.6880125 |-81.90594389|
+----+--------------------+----------------+-----+-------+-----------+------------+
only showing top 5 rows
+----+-----------------------------+
|Code|Description                  |
+----+-----------------------------+
|02Q |Titan Airways                |
|04Q |Tradewind Aviation           |
|05Q |Comlux Aviation, AG          

Calidad de vuelos

In [12]:
from pyspark.sql import functions as F

vuelo_valido = (
    (F.col("Cancelled") == 0)
    & (F.col("Diverted") == 0)
    & F.col("ArrDelay").isNotNull()
)

resumen_calidad = flights.agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("Cancelled") == 1, 1).otherwise(0)).alias("cancelados"),
    F.sum(F.when(F.col("Diverted") == 1, 1).otherwise(0)).alias("desviados"),
    F.sum(F.when(F.col("ArrDelay").isNull(), 1).otherwise(0)).alias("sin_arrdelay"),
    F.sum(F.when(vuelo_valido, 1).otherwise(0)).alias("validos"),
)

resumen_calidad.show(truncate=False)

+-------+----------+---------+------------+-------+
|total  |cancelados|desviados|sin_arrdelay|validos|
+-------+----------+---------+------------+-------+
|5411843|97763     |12081    |109844      |5301999|
+-------+----------+---------+------------+-------+



Crear label y filtrar vuelos validos

In [13]:
flights_valid = flights.filter(vuelo_valido)

flights_ml = flights_valid.withColumn(
    "Delayed",
    (F.col("ArrDelay") > 15).cast("int"),
)

flights_ml.select(
    "ArrDelay", "Delayed", "Cancelled", "Diverted"
).show(10)

+--------+-------+---------+--------+
|ArrDelay|Delayed|Cancelled|Diverted|
+--------+-------+---------+--------+
|       3|      0|        0|       0|
|      10|      0|        0|       0|
|       5|      0|        0|       0|
|       4|      0|        0|       0|
|      44|      1|        0|       0|
|      -3|      0|        0|       0|
|       0|      0|        0|       0|
|      15|      0|        0|       0|
|      11|      0|        0|       0|
|       9|      0|        0|       0|
+--------+-------+---------+--------+
only showing top 10 rows


### Comprobar estructura y conteos de 2001 y 2002

In [22]:
import csv
import re
from collections import Counter
from pathlib import Path

byte_no_utf8 = re.compile(r"[\udc80-\udcff]")
columnas_esperadas = schema_vuelos.fieldNames()

for anio in (2001, 2002):
    ruta = Path(archivos_por_anio[anio])

    total = 0
    estructura_valida = 0
    cuarentena_estructural = 0
    filas_con_bytes_no_utf8 = 0
    solo_tailnum = 0
    campos_afectados = Counter()
    ejemplos_cuarentena = []

    with ruta.open(
        "r", encoding="utf-8", errors="surrogateescape", newline=""
    ) as contenido:
        lector = csv.reader(contenido, strict=True)
        cabecera = next(lector)

        if cabecera != columnas_esperadas:
            raise ValueError(f"{anio}: la cabecera no coincide con el esquema")

        while True:
            linea_inicio = lector.line_num + 1

            try:
                fila = next(lector)
            except StopIteration:
                break
            except csv.Error as error:
                raise RuntimeError(
                    f"{anio}: error al interpretar el CSV cerca de "
                    f"la línea {linea_inicio}. No se puede cerrar el conteo."
                ) from error

            total += 1

            if len(fila) != len(cabecera):
                cuarentena_estructural += 1
                if len(ejemplos_cuarentena) < 3:
                    ejemplos_cuarentena.append(
                        (linea_inicio, len(fila))
                    )
                continue

            estructura_valida += 1

            campos_malos = [
                cabecera[i]
                for i, valor in enumerate(fila)
                if not valor.isascii() and byte_no_utf8.search(valor)
            ]

            if campos_malos:
                filas_con_bytes_no_utf8 += 1
                campos_afectados.update(campos_malos)
                if campos_malos == ["TailNum"]:
                    solo_tailnum += 1

    print(f"\nAño {anio}")
    print("Registros de entrada:", total)
    print("Estructura válida:", estructura_valida)
    print("Cuarentena estructural:", cuarentena_estructural)
    print(
        "Conteo cuadra:",
        total == estructura_valida + cuarentena_estructural,
    )
    print("Con bytes no UTF-8 y estructura válida:", filas_con_bytes_no_utf8)
    print("Afectan solo a TailNum:", solo_tailnum)
    print("Campos afectados:", dict(campos_afectados))
    print("Ejemplos de estructura dañada:", ejemplos_cuarentena)


Año 2001
Registros de entrada: 5967780
Estructura válida: 5967780
Cuarentena estructural: 0
Conteo cuadra: True
Con bytes no UTF-8 y estructura válida: 2545013
Afectan solo a TailNum: 2545013
Campos afectados: {'TailNum': 2545013}
Ejemplos de estructura dañada: []

Año 2002
Registros de entrada: 5271359
Estructura válida: 5271359
Cuarentena estructural: 0
Conteo cuadra: True
Con bytes no UTF-8 y estructura válida: 335925
Afectan solo a TailNum: 335925
Campos afectados: {'TailNum': 335925}
Ejemplos de estructura dañada: []


### Comprobar la lectura y los vuelos válidos

In [23]:
from pyspark.sql import functions as F

for anio in (2001, 2002):
    vuelos_anio = (
        spark.read
        .schema(schema_vuelos)
        .option("header", True)
        .option("enforceSchema", False)
        .option("nullValue", "NA")
        .csv(str(archivos_por_anio[anio]))
    )

    vuelo_valido = (
        (F.col("Cancelled") == 0)
        & (F.col("Diverted") == 0)
        & F.col("ArrDelay").isNotNull()
    )

    resumen = vuelos_anio.agg(
        F.count("*").alias("filas_spark"),
        F.sum(F.when(
            F.col("Year").isNull() | (F.col("Year") != anio), 1
        ).otherwise(0)).alias("year_invalido"),
        F.sum(F.when(
            F.col("Cancelled").isNull() | F.col("Diverted").isNull(), 1
        ).otherwise(0)).alias("estado_no_interpretable"),
        F.sum(F.when(
            F.col("ArrDelay").isNull(), 1
        ).otherwise(0)).alias("arrdelay_no_disponible"),
        F.sum(F.when(
            vuelo_valido, 1
        ).otherwise(0)).alias("vuelos_validos"),
    ).first()

    print(f"{anio}: {resumen.asDict()}")

2001: {'filas_spark': 5967780, 'year_invalido': 0, 'estado_no_interpretable': 0, 'arrdelay_no_disponible': 244107, 'vuelos_validos': 5723673}


Task was destroyed but it is pending!
task: <Task pending name='Task-304' coro=<_async_in_context.<locals>.run_in_context() done, defined at /opt/conda/lib/python3.13/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-305' coro=<Kernel.shell_main() running at /opt/conda/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/conda/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py:564]>
/usr/local/spark/python/pyspark/errors/utils.py:280: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  selected_frames = (
Task was destroyed but it is pending!
task: <Task pending name='Task-305' coro=<Kernel.shell_main() running at /opt/conda/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>
Task was destroyed but it is pending!
task: <Task pending name='Task-341' coro=<_async_in_context.<locals>.run_in_context() done, defined at /opt/conda/lib/

2002: {'filas_spark': 5271359, 'year_invalido': 0, 'estado_no_interpretable': 0, 'arrdelay_no_disponible': 73499, 'vuelos_validos': 5197860}


### Definir las marcas de aptitud

In [26]:
from pyspark.sql import functions as F

COLUMNAS_TRABAJO = [
    "Year", "Month", "DayOfWeek", "CRSDepTime",
    "UniqueCarrier", "Origin", "Dest", "Distance",
    "ArrDelay", "Cancelled", "Diverted",
]

def marcar_aptitud(vuelos, anio_fuente):
    def cumple(condicion):
        # Un resultado desconocido (null) cuenta como False.
        return F.coalesce(condicion, F.lit(False))

    def texto_presente(nombre):
        return cumple(F.length(F.trim(F.col(nombre))) > 0)

    def motivo_si_falla(condicion, nombre):
        return F.when(~condicion, F.lit(nombre))

    anio_ok = cumple(F.col("Year") == anio_fuente)
    aerolinea_ok = texto_presente("UniqueCarrier")
    origen_ok = texto_presente("Origin")
    destino_ok = texto_presente("Dest")

    cancelacion_valida = cumple(F.col("Cancelled").isin(0, 1))
    desvio_valido = cumple(F.col("Diverted").isin(0, 1))
    cancelado = cumple(F.col("Cancelled") == 1)
    desviado = cumple(F.col("Diverted") == 1)
    arrdelay_disponible = F.col("ArrDelay").isNotNull()

    vuelo_valido = cumple(
        (F.col("Cancelled") == 0)
        & (F.col("Diverted") == 0)
        & arrdelay_disponible
    )

    mes_ok = cumple(F.col("Month").between(1, 12))
    dia_semana_ok = cumple(F.col("DayOfWeek").between(1, 7))
    hora_programada_ok = F.col("CRSDepTime").isNotNull()
    distancia_ok = cumple(F.col("Distance") > 0)

    motivos_comunes = [
        F.when(cancelado, F.lit("cancelado")),
        F.when(~cancelacion_valida, F.lit("cancelled_invalido")),
        F.when(desviado, F.lit("desviado")),
        F.when(~desvio_valido, F.lit("diverted_invalido")),
        motivo_si_falla(arrdelay_disponible, "arrdelay_no_disponible"),
    ]

    def juntar_motivos(*motivos_adicionales):
        # concat_ws omite los motivos que son null.
        return F.concat_ws(
            ";",
            *(motivos_comunes + list(motivos_adicionales))
        )

    motivos_anio = motivo_si_falla(anio_ok, "anio_no_coincide")
    motivos_aerolinea = motivo_si_falla(
        aerolinea_ok, "aerolinea_ausente"
    )
    motivos_origen = motivo_si_falla(origen_ok, "origen_ausente")
    motivos_destino = motivo_si_falla(destino_ok, "destino_ausente")

    predictores_ok = (
        mes_ok
        & dia_semana_ok
        & hora_programada_ok
        & distancia_ok
    )

    return (
        vuelos.select(*COLUMNAS_TRABAJO)
        .withColumn("vuelo_valido", vuelo_valido)
        .withColumn("motivos_vuelo_valido", juntar_motivos())

        .withColumn(
            "apto_consulta_1",
            vuelo_valido & anio_ok & aerolinea_ok,
        )
        .withColumn(
            "motivos_consulta_1",
            juntar_motivos(motivos_anio, motivos_aerolinea),
        )

        .withColumn(
            "apto_consulta_2",
            vuelo_valido & anio_ok & origen_ok & destino_ok,
        )
        .withColumn(
            "motivos_consulta_2",
            juntar_motivos(
                motivos_anio, motivos_origen, motivos_destino
            ),
        )

        .withColumn(
            "apto_consulta_3",
            vuelo_valido & anio_ok & aerolinea_ok & origen_ok,
        )
        .withColumn(
            "motivos_consulta_3",
            juntar_motivos(
                motivos_anio, motivos_aerolinea, motivos_origen
            ),
        )

        .withColumn(
            "apto_modelo",
            vuelo_valido & anio_ok & aerolinea_ok
            & origen_ok & destino_ok & predictores_ok,
        )
        .withColumn(
            "motivos_modelo",
            juntar_motivos(
                motivos_anio,
                motivos_aerolinea,
                motivos_origen,
                motivos_destino,
                motivo_si_falla(mes_ok, "mes_invalido"),
                motivo_si_falla(dia_semana_ok, "dia_semana_invalido"),
                motivo_si_falla(
                    hora_programada_ok, "hora_programada_ausente"
                ),
                motivo_si_falla(distancia_ok, "distancia_invalida"),
            ),
        )
    )

### Comprobar las marcas con cinco ejemplos

In [27]:
from pyspark.sql.types import StructType

schema_prueba = StructType([
    schema_vuelos[nombre] for nombre in COLUMNAS_TRABAJO
])

prueba = spark.createDataFrame(
    [
        # Apto para todas las salidas.
        (2001, 1, 2, 900, "US", "AAA", "BBB", 300, 20, 0, 0),

        # Sin Distance: consultas sí, modelo no.
        (2001, 1, 2, 900, "US", "BBB", "CCC", None, 20, 0, 0),

        # Sin aerolínea: solo sirve para la consulta de rutas.
        (2001, 1, 2, 900, " ", "CCC", "DDD", 300, 20, 0, 0),

        # Cancelado: no sirve para analizar retrasos.
        (2001, 1, 2, 900, "US", "DDD", "EEE", 300, 20, 1, 0),

        # Year no coincide con el año del archivo.
        (2002, 1, 2, 900, "US", "EEE", "FFF", 300, 20, 0, 0),
    ],
    schema=schema_prueba,
)

marcas = [
    "vuelo_valido",
    "apto_consulta_1",
    "apto_consulta_2",
    "apto_consulta_3",
    "apto_modelo",
]

motivos = [
    "motivos_vuelo_valido",
    "motivos_consulta_1",
    "motivos_consulta_2",
    "motivos_consulta_3",
    "motivos_modelo",
]

filas_prueba = (
    marcar_aptitud(prueba, anio_fuente=2001)
    .select("Origin", *marcas, *motivos)
    .orderBy("Origin")
    .collect()
)

for fila in filas_prueba:
    print(fila.asDict())

marcas_observadas = {
    fila["Origin"]: tuple(fila[nombre] for nombre in marcas)
    for fila in filas_prueba
}

motivos_observados = {
    fila["Origin"]: tuple(fila[nombre] for nombre in motivos)
    for fila in filas_prueba
}

marcas_esperadas = {
    "AAA": (True,  True,  True,  True,  True),
    "BBB": (True,  True,  True,  True,  False),
    "CCC": (True,  False, True,  False, False),
    "DDD": (False, False, False, False, False),
    "EEE": (True,  False, False, False, False),
}

motivos_esperados = {
    "AAA": ("", "", "", "", ""),
    "BBB": ("", "", "", "", "distancia_invalida"),
    "CCC": (
        "",
        "aerolinea_ausente",
        "",
        "aerolinea_ausente",
        "aerolinea_ausente",
    ),
    "DDD": ("cancelado",) * 5,
    "EEE": ("", "anio_no_coincide") + ("anio_no_coincide",) * 3,
}

assert marcas_observadas == marcas_esperadas, "Fallaron las marcas"
assert motivos_observados == motivos_esperados, "Fallaron los motivos"

for fila in filas_prueba:
    for marca, motivo in zip(marcas, motivos):
        assert fila[marca] == (fila[motivo] == ""), (
            f"Incoherencia en {fila['Origin']}: {marca} / {motivo}"
        )

print("OK: marcas y motivos coinciden con las reglas")

{'Origin': 'AAA', 'vuelo_valido': True, 'apto_consulta_1': True, 'apto_consulta_2': True, 'apto_consulta_3': True, 'apto_modelo': True, 'motivos_vuelo_valido': '', 'motivos_consulta_1': '', 'motivos_consulta_2': '', 'motivos_consulta_3': '', 'motivos_modelo': ''}
{'Origin': 'BBB', 'vuelo_valido': True, 'apto_consulta_1': True, 'apto_consulta_2': True, 'apto_consulta_3': True, 'apto_modelo': False, 'motivos_vuelo_valido': '', 'motivos_consulta_1': '', 'motivos_consulta_2': '', 'motivos_consulta_3': '', 'motivos_modelo': 'distancia_invalida'}
{'Origin': 'CCC', 'vuelo_valido': True, 'apto_consulta_1': False, 'apto_consulta_2': True, 'apto_consulta_3': False, 'apto_modelo': False, 'motivos_vuelo_valido': '', 'motivos_consulta_1': 'aerolinea_ausente', 'motivos_consulta_2': '', 'motivos_consulta_3': 'aerolinea_ausente', 'motivos_modelo': 'aerolinea_ausente'}
{'Origin': 'DDD', 'vuelo_valido': False, 'apto_consulta_1': False, 'apto_consulta_2': False, 'apto_consulta_3': False, 'apto_modelo': F

### Resumir las marcas en 2001 y 2002

In [28]:
from pyspark.sql import functions as F

nombres_marcas = [
    "vuelo_valido",
    "apto_consulta_1",
    "apto_consulta_2",
    "apto_consulta_3",
    "apto_modelo",
]

preparados_por_anio = {}
resumen_por_anio = {}

for anio in (2001, 2002):
    vuelos = (
        spark.read
        .schema(schema_vuelos)
        .option("header", True)
        .option("enforceSchema", False)
        .option("nullValue", "NA")
        .csv(str(archivos_por_anio[anio]))
    )

    preparados = marcar_aptitud(vuelos, anio)
    preparados_por_anio[anio] = preparados

    resumen = preparados.agg(
        F.count("*").alias("filas"),
        *[
            F.sum(F.col(nombre).cast("long")).alias(nombre)
            for nombre in nombres_marcas
        ],
    ).first().asDict()

    resumen_por_anio[anio] = resumen
    print(f"{anio}: {resumen}")

2001: {'filas': 5967780, 'vuelo_valido': 5723673, 'apto_consulta_1': 5723673, 'apto_consulta_2': 5723673, 'apto_consulta_3': 5723673, 'apto_modelo': 5723673}
2002: {'filas': 5271359, 'vuelo_valido': 5197860, 'apto_consulta_1': 5197860, 'apto_consulta_2': 5197860, 'apto_consulta_3': 5197860, 'apto_modelo': 5197860}


## 8 · Preparar vuelos de todos los años

In [29]:
from pyspark.sql import functions as F
from pyspark.sql import Window

anios = sorted(archivos_por_anio)
assert anios == list(range(1987, 2009)), f"Años encontrados: {anios}"

vuelos_todos = None

for anio in anios:
    vuelos_anio = (
        spark.read
        .schema(schema_vuelos)
        .option("header", True)
        .option("enforceSchema", False)
        .option("nullValue", "NA")
        .csv(str(archivos_por_anio[anio]))
    )

    preparados = (
        marcar_aptitud(vuelos_anio, anio)
        .withColumn("_anio_fuente", F.lit(anio))
    )

    vuelos_todos = (
        preparados if vuelos_todos is None
        else vuelos_todos.unionByName(preparados)
    )

print("Plan preparado para", len(anios), "años")

Plan preparado para 22 años


## 9 · Consulta 1: retraso medio por aerolínea y año

In [30]:
consulta_1 = (
    vuelos_todos
    .filter(F.col("apto_consulta_1"))
    .groupBy("Year", "UniqueCarrier")
    .agg(
        F.count("*").alias("n_vuelos"),
        F.avg("ArrDelay").alias("retraso_medio"),
    )
    .filter(F.col("n_vuelos") >= 100)
)

resultado_1 = consulta_1.orderBy("Year", "UniqueCarrier").collect()

print("Años en el resultado:", sorted({fila["Year"] for fila in resultado_1}))
print("Grupos aerolínea-año:", len(resultado_1))

for fila in resultado_1[:15]:
    print(fila.asDict())

Años en el resultado: [1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008]
Grupos aerolínea-año: 295
{'Year': 1987, 'UniqueCarrier': 'AA', 'n_vuelos': 162609, 'retraso_medio': 4.914672619596701}
{'Year': 1987, 'UniqueCarrier': 'AS', 'n_vuelos': 21141, 'retraso_medio': 12.430301310250226}
{'Year': 1987, 'UniqueCarrier': 'CO', 'n_vuelos': 119978, 'retraso_medio': 11.17253163079898}
{'Year': 1987, 'UniqueCarrier': 'DL', 'n_vuelos': 183639, 'retraso_medio': 11.92380158898709}
{'Year': 1987, 'UniqueCarrier': 'EA', 'n_vuelos': 105564, 'retraso_medio': 9.150126937213445}
{'Year': 1987, 'UniqueCarrier': 'HP', 'n_vuelos': 44781, 'retraso_medio': 9.538308657689646}
{'Year': 1987, 'UniqueCarrier': 'NW', 'n_vuelos': 106088, 'retraso_medio': 10.716904833722946}
{'Year': 1987, 'UniqueCarrier': 'PA (1)', 'n_vuelos': 16545, 'retraso_medio': 7.423813841039589}
{'Year': 1987, 'UniqueCarrier': 'PI', 'n_vuelos': 115110, 'retra

## 10 · Consulta 2: tres rutas con más retraso por origen

In [32]:
rutas = (
    vuelos_todos
    .filter(F.col("apto_consulta_2"))
    .groupBy("Origin", "Dest")
    .agg(
        F.count("*").alias("n_vuelos"),
        F.avg("ArrDelay").alias("retraso_medio"),
    )
    .filter(F.col("n_vuelos") >= 100)
)

ventana_rutas = Window.partitionBy("Origin").orderBy(
    F.col("retraso_medio").desc(),
    F.col("Dest").asc(),  # Desempate reproducible
)

consulta_2 = (
    rutas
    .withColumn("puesto", F.row_number().over(ventana_rutas))
    .filter(F.col("puesto") <= 3)
)

consulta_2.orderBy("Origin", "puesto").show(30, truncate=False)

+------+----+--------+--------------------+------+
|Origin|Dest|n_vuelos|retraso_medio       |puesto|
+------+----+--------+--------------------+------+
|ABE   |PHL |497     |12.845070422535212  |1     |
|ABE   |ORD |22566   |8.445005760879198   |2     |
|ABE   |ATL |15513   |7.602397988783601   |3     |
|ABI   |DFW |17816   |2.7441625505163896  |1     |
|ABI   |IAH |1596    |-1.4141604010025062 |2     |
|ABQ   |SFO |10284   |11.28432516530533   |1     |
|ABQ   |TUS |10300   |8.048155339805826   |2     |
|ABQ   |AMA |14960   |6.811697860962567   |3     |
|ABY   |ATL |7104    |8.474802927927929   |1     |
|ACK   |EWR |1065    |30.11830985915493   |1     |
|ACK   |JFK |132     |29.022727272727273  |2     |
|ACT   |DFW |17066   |-0.06369389429274581|1     |
|ACT   |IAH |2189    |-1.135678391959799  |2     |
|ACV   |SFO |14226   |12.464712498242655  |1     |
|ACV   |SMF |3235    |2.170015455950541   |2     |
|ACV   |CEC |2829    |0.19052668787557442 |3     |
|ACY   |ATL |940     |20.938297

## 11 · Consulta 3: aerolínea frente a su aeropuerto de origen

In [31]:
base_origen = vuelos_todos.filter(
    F.col("vuelo_valido")
    & (F.col("Year") == F.col("_anio_fuente"))
    & (F.length(F.trim(F.col("Origin"))) > 0)
)

por_aerolinea_origen = (
    base_origen
    .groupBy("Origin", "UniqueCarrier")
    .agg(
        F.count("*").alias("n_vuelos"),
        F.sum("ArrDelay").alias("suma_retrasos"),
    )
)

ventana_origen = Window.partitionBy("Origin")

consulta_3 = (
    por_aerolinea_origen
    .withColumn(
        "vuelos_origen",
        F.sum("n_vuelos").over(ventana_origen),
    )
    .withColumn(
        "suma_retrasos_origen",
        F.sum("suma_retrasos").over(ventana_origen),
    )
    .filter(
        (F.col("UniqueCarrier").isNotNull())
        & (F.length(F.trim(F.col("UniqueCarrier"))) > 0)
        & (F.col("n_vuelos") >= 100)
    )
    .withColumn(
        "retraso_medio_aerolinea",
        F.col("suma_retrasos") / F.col("n_vuelos"),
    )
    .withColumn(
        "retraso_medio_origen",
        F.col("suma_retrasos_origen") / F.col("vuelos_origen"),
    )
    .withColumn(
        "diferencia_minutos",
        F.col("retraso_medio_aerolinea")
        - F.col("retraso_medio_origen"),
    )
    .select(
        "Origin", "UniqueCarrier", "n_vuelos",
        "retraso_medio_aerolinea",
        "retraso_medio_origen",
        "diferencia_minutos",
    )
)

consulta_3.orderBy(
    F.col("diferencia_minutos").desc()
).show(20, truncate=False)

+------+-------------+--------+-----------------------+--------------------+------------------+
|Origin|UniqueCarrier|n_vuelos|retraso_medio_aerolinea|retraso_medio_origen|diferencia_minutos|
+------+-------------+--------+-----------------------+--------------------+------------------+
|GEG   |PS           |170     |38.141176470588235     |5.000120279529627   |33.14105619105861 |
|BGR   |XE           |1476    |37.33807588075881      |6.028685165671467   |31.30939071508734 |
|SCE   |EV           |314     |25.127388535031848     |-1.8648194794290511 |26.9922080144609  |
|EWR   |OO           |1049    |36.8884652049571       |10.047035606208976  |26.841429598748125|
|BDL   |ML (1)       |172     |30.523255813953487     |4.831036552709457   |25.69221926124403 |
|BHM   |MQ           |666     |29.984984984984983     |5.541739927643693   |24.44324505734129 |
|CID   |EV           |649     |28.724191063174114     |5.576201636306719   |23.147989426867397|
|SEA   |PA (1)       |310     |29.267741